# TP 7 — CNN 1D Classification de type de culture avec **Keras/TensorFlow**

*Notebook miroir de TP 8 (PyTorch) — même cas d'usage, même dataset, API différente.*

## Introduction

Les **Convolutional Neural Networks en 1 dimension (CNN 1D)** sont particulierement adaptes
pour extraire des motifs locaux dans des sequences de features — ce qui les rend utiles
pour la classification de donnees tabulaires agronomiques (sequences de mesures de sol, climat,
superficie).

**Cas d'usage** : Classifier le type de saison agricole (`Kharif` / `Rabi` / `Whole Year`)
a partir des features numeriques du dataset crop yield.

**Objectifs pedagogiques :**
- Preparer des donnees tabulaires pour un CNN 1D (reshape en sequences)
- Construire une architecture `Conv1D -> MaxPool -> Conv1D -> Dense` avec Keras
- Comprendre comment les filtres convolutionnels extraient des motifs locaux dans les features
- Comparer avec TP 8 (meme architecture en PyTorch)

**Difference Keras vs PyTorch (cf. TP 8) :**
- Keras : `Conv1D` attend `(batch, steps, features)` — le padding est un parametre de la couche
- PyTorch : `nn.Conv1d` attend `(batch, features, steps)` — les dimensions sont transposees

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

print('TensorFlow version:', tf.__version__)

## 1. Chargement et preparation des donnees

In [ ]:
df = pd.read_excel('Data/crop_csv_file.xlsx')
print('Shape:', df.shape)
df.head(3)

## 2. Feature engineering et encodage

On selectionne les features numeriques pour classifier le type de saison :
- Features : `Temperature`, `humidity`, `soil moisture`, `area`, `Crop_Year`
- Cible : `Season` (Kharif / Rabi / Whole Year / Autumn / Summer / Winter)

In [ ]:
# Nettoyage : strip des espaces dans la colonne Season
df['Season'] = df['Season'].str.strip()

# Selection des features numeriques
FEATURES = ['Temperature', 'humidity', 'soil moisture', ' area', 'Crop_Year', 'Production']
TARGET   = 'Season'

# Suppression des lignes avec NaN
df_clean = df[FEATURES + [TARGET]].dropna()
print('Apres nettoyage:', df_clean.shape)
print('Distribution des classes:')
print(df_clean[TARGET].value_counts())

In [ ]:
# Encodage de la cible
le = LabelEncoder()
y_encoded = le.fit_transform(df_clean[TARGET])
NUM_CLASSES = len(le.classes_)
print('Classes:', le.classes_)
print('Nombre de classes:', NUM_CLASSES)

# Normalisation des features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[FEATURES])

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## 3. Reshape pour CNN 1D

Le CNN 1D attend des donnees de forme `(samples, steps, channels)`.
On traite chaque feature comme un 'pas de temps' et on a 1 canal.

**Keras** : `(batch, steps, features)` = `(n, 6, 1)`
**PyTorch** (TP 8) : `(batch, channels, length)` = `(n, 1, 6)` — les dimensions sont inversees !

In [ ]:
# Reshape pour Keras Conv1D : (samples, steps, channels)
N_STEPS = X_train.shape[1]  # = 6 features
X_train_cnn = X_train.reshape(X_train.shape[0], N_STEPS, 1)
X_test_cnn  = X_test.reshape(X_test.shape[0], N_STEPS, 1)

# One-hot encoding de la cible pour Keras
y_train_oh = to_categorical(y_train, num_classes=NUM_CLASSES)
y_test_oh  = to_categorical(y_test, num_classes=NUM_CLASSES)

print('X_train_cnn shape:', X_train_cnn.shape)  # (n, 6, 1)
print('y_train_oh  shape:', y_train_oh.shape)    # (n, NUM_CLASSES)

## 4. Architecture CNN 1D (Keras)

```
Input (steps=6, channels=1)
  |
  v
Conv1D(32, kernel_size=2, activation='relu')
  |
BatchNormalization
  |
Conv1D(64, kernel_size=2, activation='relu')
  |
GlobalAveragePooling1D     <- reduit (batch, steps, 64) en (batch, 64)
  |
Dropout(0.3)
  |
Dense(64, relu) -> Dense(NUM_CLASSES, softmax)
```

In [ ]:
model = Sequential([
    Conv1D(filters=32, kernel_size=2, activation='relu',
           padding='same', input_shape=(N_STEPS, 1)),
    BatchNormalization(),
    Conv1D(filters=64, kernel_size=2, activation='relu', padding='same'),
    GlobalAveragePooling1D(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax')
], name='CNN1D_Crop_Classification_Keras')

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

## 5. Entrainement

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train_cnn, y_train_oh,
    epochs=80,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Courbes d'apprentissage
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['loss'], label='Train Loss', color='#e74c3c')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#3498db')
axes[0].set_title('Loss — CNN 1D Keras')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Train Acc', color='#e74c3c')
axes[1].plot(history.history['val_accuracy'], label='Val Acc', color='#3498db')
axes[1].set_title('Accuracy — CNN 1D Keras')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print('Epochs effectuees:', len(history.history['loss']))

## 6. Evaluation

In [ ]:
# Evaluation sur le jeu de test
test_loss, test_acc = model.evaluate(X_test_cnn, y_test_oh, verbose=0)
print('Test Accuracy:', round(test_acc * 100, 2), '%')

# Predictions
y_pred_proba = model.predict(X_test_cnn)
y_pred = np.argmax(y_pred_proba, axis=1)

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Matrice de confusion — CNN 1D Keras', fontsize=14)
plt.ylabel('Reel'); plt.xlabel('Predit')
plt.tight_layout(); plt.show()

## 7. Resume comparatif Keras vs PyTorch

| Aspect | Keras (ce notebook) | PyTorch (TP 8) |
|---|---|---|
| **Format entree** | `(batch, steps, channels)` = `(n, 6, 1)` | `(batch, channels, length)` = `(n, 1, 6)` |
| **Convolution** | `Conv1D(32, kernel_size=2)` | `nn.Conv1d(1, 32, kernel_size=2)` |
| **Padding** | `padding='same'` (parametre) | `padding=1` (nombre de pixels) |
| **Pooling global** | `GlobalAveragePooling1D()` | `nn.AdaptiveAvgPool1d(1)` + `.squeeze()` |
| **Cible** | One-hot + `categorical_crossentropy` | Indices + `nn.CrossEntropyLoss` (integre softmax) |
| **Batch norm** | `BatchNormalization()` | `nn.BatchNorm1d(channels)` |